In [1]:
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr
import cartopy.crs as ccrs
from heatextremes.metrics import coverage, mean_in_time_batches, probability_of_exceedance_brier_score
import heatextremes as he 
from dask.diagnostics import ProgressBar 
from dask.distributed import Client
from pathlib import Path 
import pandas as pd 

import warnings
from zarr.errors import ZarrUserWarning

days = [1,3,5]
years = [2022, 2023, 2024, 2025]


In [2]:
root = Path("/net/monsoon/marchakitus/AIFS/v2p0/combined/forecasts_AIFS_ENS_v2")
paths = sorted(root.glob("*.zarr"))
paths = [_ for _ in paths if pd.Timestamp(_.name[5:-5]).year in years] 
wanted = ["2t"]


In [3]:
with warnings.catch_warnings():
    warnings.filterwarnings(
        "ignore",
        message=r"Numcodecs codecs are not in the Zarr version 3 specification.*",
        category=ZarrUserWarning,
    )
    with ProgressBar():
        ds = xr.open_mfdataset(
            paths,
            engine="zarr",
            combine="nested",
            concat_dim="time",
            preprocess=lambda x: x[wanted],
            chunks = {
                "time": 4,  # unavoidable: one time per store
                "number": 26,  # combine all ensemble members
                "prediction_timedelta": 24,
                "latitude": 360,
                "longitude": 720,
            },
            parallel=True,
            data_vars="all",
            coords="minimal",
            compat="override",
            join="override",
            combine_attrs="override",
            consolidated=None,
        )

[########################################] | 100% Completed | 32.70 s


In [4]:
sixhourly_leads = [pd.Timedelta(i) for i in ds.prediction_timedelta.values if pd.Timedelta(i).days in days]
t2m_daily_max = ds['2t'].sel(prediction_timedelta=sixhourly_leads).groupby("prediction_timedelta.days").max().rename({'lat': 'latitude', 'lon': 'longitude'})

In [14]:
t2m_daily_max = t2m_daily_max.chunk({
    "time": 36,                 # use your actual initialization-time dimension
    "number": -1,
    "latitude": 360,
    "longitude": 720,
})

In [ ]:
with ProgressBar():
    t2m_daily_max.to_dataset(name="t2m_daily_max").to_zarr(
        "/net/monsoon/kylehall/aifs_ensv2_t2m_daily_max.zarr",
        mode="w",
    )

/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


[########################################] | 100% Completed | 85m 53s


In [5]:
era5 = he.open_cached_era5(chunks = {"time": 36, "latitude": 360, "longitude": 720}) 
era5 = era5['2m_temperature']

/home/kylehall/heat-extremes/heatextremes/open_cached_era5.py:64: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 36. This could degrade performance. Instead, consider rechunking after loading.
  xr.open_zarr(path, consolidated=True, chunks=dask_chunks)
/home/kylehall/heat-extremes/heatextremes/open_cached_era5.py:64: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 36. This could degrade performance. Instead, consider rechunking after loading.
  xr.open_zarr(path, consolidated=True, chunks=dask_chunks)
/home/kylehall/heat-extremes/heatextremes/open_cached_era5.py:64: UserWarning: The specified chunks separate the stored chunks along dimension "time" starting at index 36. This could degrade performance. Instead, consider rechunking after loading.
  xr.open_zarr(path, consolidated=True, chunks=dask_chunks)
/home/kylehall/heat-extremes/heatextremes/open_cached_era5.py:64: UserWarnin

In [6]:
sixhourly_leads_obs = [ [ pd.Timedelta(i) for i in ds.prediction_timedelta.values if pd.Timedelta(i).days == days[j] ] for j in range(3)]

with ProgressBar():
    era5_with_leads = xr.concat([
        xr.concat([
            era5.sel(time = t2m_daily_max.time + sixhourly_leads_obs[i][j] ).drop('time') for j in range(4)
        ], 'time_in_day').max('time_in_day') for i in range(len(sixhourly_leads_obs))
    ], 'days').assign_coords(days=days)

era5_with_leads

/tmp/ipykernel_67681/2771630230.py:6: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5.sel(time = t2m_daily_max.time + sixhourly_leads_obs[i][j] ).drop('time') for j in range(4)
/tmp/ipykernel_67681/2771630230.py:6: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5.sel(time = t2m_daily_max.time + sixhourly_leads_obs[i][j] ).drop('time') for j in range(4)
/tmp/ipykernel_67681/2771630230.py:6: FutureWarning: dropping variables using `drop` is deprecated; use drop_vars.
  era5.sel(time = t2m_daily_max.time + sixhourly_leads_obs[i][j] ).drop('time') for j in range(4)


<xarray.DataArray '2m_temperature' (days: 3, time: 364, latitude: 721,
                                    longitude: 1440)> Size: 5GB
dask.array<concatenate, shape=(3, 364, 721, 1440), dtype=float32, chunksize=(1, 35, 360, 720), chunktype=numpy.ndarray>
Coordinates:
  * days       (days) int64 24B 1 3 5
  * latitude   (latitude) float64 6kB -90.0 -89.75 -89.5 ... 89.5 89.75 90.0
  * longitude  (longitude) float64 12kB -180.0 -179.8 -179.5 ... 179.5 179.8
Dimensions without coordinates: time
Attributes: (12/30)
    GRIB_NV:                                  0
    GRIB_Nx:                                  1440
    GRIB_Ny:                                  721
    GRIB_cfName:                              unknown
    GRIB_cfVarName:                           t2m
    GRIB_dataType:                            an
    ...                                       ...
    GRIB_totalNumber:                         0
    GRIB_typeOfLevel:                         surface
    GRIB_units:                               K
    long_name:                                2 metre temperature
    standard_name:                            unknown
    units:                                    K

In [7]:
t2m_daily_max = xr.open_zarr("/net/monsoon/kylehall/aifs_ensv2_t2m_daily_max.zarr", chunks={}, consolidated=True)
t2m_daily_max  = t2m_daily_max['t2m_daily_max']
t2m_daily_max 

<xarray.DataArray 't2m_daily_max' (time: 364, number: 26, days: 3,
                                   latitude: 721, longitude: 1440)> Size: 118GB
dask.array<open_dataset-t2m_daily_max, shape=(364, 26, 3, 721, 1440), dtype=float32, chunksize=(36, 26, 1, 360, 720), chunktype=numpy.ndarray>
Coordinates:
  * time       (time) datetime64[ns] 3kB 2022-01-01 2022-01-07 ... 2025-12-31
  * number     (number) int64 208B 0 1 2 3 4 5 6 7 8 ... 18 19 20 21 22 23 24 25
  * days       (days) int64 24B 1 3 5
  * latitude   (latitude) float64 6kB 90.0 89.75 89.5 ... -89.5 -89.75 -90.0
  * longitude  (longitude) float64 12kB 0.0 0.25 0.5 0.75 ... 359.2 359.5 359.8

In [8]:
def normalize_longitude(data: xr.Dataset | xr.DataArray) -> xr.Dataset | xr.DataArray:
    """Convert a longitude coordinate from 0--360 to [-180, 180) and sort it."""
    if "longitude" not in data.coords:
        raise ValueError("All inputs must have a longitude coordinate")
    normalized_longitude = (data.longitude + 180) % 360 - 180
    if normalized_longitude.to_index().has_duplicates:
        raise ValueError("Longitude normalization produced duplicate coordinates")
    return data.assign_coords(longitude=normalized_longitude).sortby("longitude")

era5_with_leads = normalize_longitude(era5_with_leads)
t2m_daily_max = normalize_longitude(t2m_daily_max)



In [9]:
def calculate_coverage(
        model: xr.DataArray | None = None, 
        obs: xr.DataArray | None = None, 
        percentile: list | None = 90, #[10, 20, 30, 40, 50, 60, 70, 80, 90],
        ensemble_member_dim: str | None = 'number'
    ):

    #percentile = np.asarray(percentile)
    lower_quantile = (100.0 - percentile) / 200.0
    upper_quantile = 1.0 - lower_quantile
    print('1')
   # model = model.chunk({ensemble_member_dim: -1})
    interval = model.quantile(
        [lower_quantile, upper_quantile], dim=ensemble_member_dim
    )

    print('2')
    lower_bound = interval.isel(quantile=0, drop=True)
    upper_bound = interval.isel(quantile=1, drop=True)
    print('3')

    # Put ERA5 dimensions in the same order as the model bounds.
    obs = obs.transpose(*lower_bound.dims)

    # Make ERA5 use exactly the bounds' chunk boundaries.
    obs = obs.chunk({
        dim: lower_bound.chunksizes[dim]
        for dim in obs.dims
    })
    print('4')
    is_covered = (lower_bound <= obs) & (
        obs <= upper_bound
    )
    print('5')
    has_valid_values = (
        obs.notnull()
        & lower_bound.notnull()
        & upper_bound.notnull()
    )
    result = is_covered.where(has_valid_values).astype(float)
    print('6')
    return result

coverage = calculate_coverage(t2m_daily_max, era5_with_leads)


1
2
3
4
5
6


In [13]:
coverage = coverage.chunk({
    "time": 36,
    'days': 1,
    "latitude": 180,
    "longitude": 720,
})


In [ ]:
import dask
with dask.config.set(scheduler="threads", num_workers=4):
    with ProgressBar():
        coverage.to_dataset(name="coverage").to_zarr(
            "/net/monsoon/kylehall/aifs_ensv2_t2m_daily_max_coverage_90.zarr",
            mode="w",
        )


/home/kylehall/miniconda3/envs/heat-extremes/lib/python3.11/site-packages/zarr/api/asynchronous.py:247: ZarrUserWarning: Consolidated metadata is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  warnings.warn(


[##################                      ] | 47% Completed | 478.15 s

## Calculate per-case scores

Each output retains `time`, `prediction_timedelta`, `latitude`, and `longitude`; only `number` is reduced. This leaves the choice of aggregation—by lead, initialization, season, or spatial region—to downstream code.

## Aggregate and map

These maps average over both initialization and lead time. Change `aggregation_dimensions` to preserve one or both dimensions, or use `groupby` for a different diagnostic.

In [ ]:
figure, axes = plt.subplots(
    ncols=1, figsize=(16, 5), subplot_kw={"projection": ccrs.PlateCarree()}
)

coverage.plot(
    ax=axes[0],
    x="longitude",
    y="latitude",
  #  transform=ccrs.PlateCarree(),
    cmap="viridis",
    vmin=0,
    vmax=1,
    cbar_kwargs={"label": "90% interval coverage"},
)
for axis, title in zip(
    axes,
    ("Mean central-90% ensemble coverage"),
):
    axis.set_global()
    axis.coastlines(linewidth=0.7)
   # gridlines = axis.gridlines(draw_labels=True, linewidth=0.4, color="gray", alpha=0.5)
   # gridlines.top_labels = False
   # gridlines.right_labels = False
    axis.set_title(title)

figure.tight_layout()


## Example: aggregate by lead time

Because the case-wise results still have `time` and `prediction_timedelta`, aggregating over space and initialization produces lead-time diagnostics without recalculating the metrics.

In [ ]:
lead_days = mean_coverage_by_lead.prediction_timedelta / np.timedelta64(1, "D")

figure, axis = plt.subplots(figsize=(8, 4))
axis.plot(lead_days, mean_coverage_by_lead, marker="o", label="90% coverage")
axis.plot(lead_days, mean_brier_by_lead, marker="o", label="Exceedance Brier score")
axis.set(xlabel="Lead time (days)", ylabel="Mean score", ylim=(0, 1))
axis.grid()
axis.legend()
figure.tight_layout()
